# eICU Demo — RMAB Transition Matrix Pipeline (Model B)

Since we're still waiting on MIMIC-IV full access, we're using the **eICU-CRD Demo v2.0.1** instead — it's fully open access, no approval needed.

The demo has around 2,520 ICU unit stays across 20 US hospitals (2014–2015), which gives us a lot more transitions to work with compared to the MIMIC demo.

The pipeline is basically the same as before — we learn two transition matrices from the data:
- **P_icu** — how patients' states evolve when they're in the ICU
- **P_nonicu** — how they evolve outside the ICU

Main difference from MIMIC: eICU uses minute offsets instead of timestamps, and vitals come from two tables (`vitalPeriodic` and `nurseCharting`).

Output: `P_icu_eicu.npy` and `P_nonicu_eicu.npy` — same 82×82 format as before.

---
## Section 1 — Setup & Data Loading

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

BASE_PATH = r'C:\Users\eric\Desktop\Mimic RMAB\data\eicu-demo'
OUT_DIR   = r'C:\Users\eric\Desktop\Mimic RMAB'

print('Base path:', BASE_PATH)
print('Exists:   ', os.path.isdir(BASE_PATH))
print('Files:    ', os.listdir(BASE_PATH))

In [ ]:
# ── Load tables ───────────────────────────────────────────────────────────────
patient = pd.read_csv(
    os.path.join(BASE_PATH, 'patient.csv.gz'), compression='gzip'
)

vitalPeriodic = pd.read_csv(
    os.path.join(BASE_PATH, 'vitalPeriodic.csv.gz'), compression='gzip',
    usecols=['patientunitstayid', 'observationoffset',
             'systemicmean', 'sao2', 'heartrate', 'respiration', 'temperature']
)

nurseCharting = pd.read_csv(
    os.path.join(BASE_PATH, 'nurseCharting.csv.gz'), compression='gzip',
    usecols=['patientunitstayid', 'nursingchartoffset',
             'nursingchartcelltypevallabel', 'nursingchartvalue']
)

lab = pd.read_csv(
    os.path.join(BASE_PATH, 'lab.csv.gz'), compression='gzip',
    usecols=['patientunitstayid', 'labresultoffset', 'labname', 'labresult']
)

hospital = pd.read_csv(
    os.path.join(BASE_PATH, 'hospital.csv.gz'), compression='gzip'
)

for name, df in [('patient', patient), ('vitalPeriodic', vitalPeriodic),
                 ('nurseCharting', nurseCharting), ('lab', lab), ('hospital', hospital)]:
    print(f'{name:<16} shape: {df.shape}')

---
## Section 2 — Building the Daily Backbone

eICU records time as **minute offsets from ICU admission**, not actual timestamps, so we convert everything to day indices by dividing by 1440.

- `hospitaladmitoffset` — how many minutes before ICU admit the patient entered the hospital (usually negative)
- `unitdischargeoffset` — when the patient left the ICU
- `hospitaldischargeoffset` — when the patient left the hospital

We go through each stay and label every day as ICU or non-ICU based on whether it falls within the ICU window.

In [ ]:
# Drop stays with missing discharge offsets
pat = patient.dropna(subset=['hospitaladmitoffset', 'hospitaldischargeoffset',
                              'unitdischargeoffset', 'hospitaldischargestatus']).copy()
pat['hospitaldischargeoffset'] = pat['hospitaldischargeoffset'].astype(int)
pat['unitdischargeoffset']     = pat['unitdischargeoffset'].astype(int)
pat['hospitaladmitoffset']     = pat['hospitaladmitoffset'].astype(int)

print(f'Stays after dropping NaN offsets: {len(pat):,} of {len(patient):,}')
print(f'Unique patients (uniquepid):      {pat["uniquepid"].nunique():,}')
print(f'Unique health-system stays:       {pat["patienthealthsystemstayid"].nunique():,}')
print(f'\nhospitaldischargestatus:')
print(pat['hospitaldischargestatus'].value_counts())

In [ ]:
# ── Build daily backbone ──────────────────────────────────────────────────────
MINS_PER_DAY = 1440  # 60 * 24

rows = []
for _, stay in pat.iterrows():
    uid  = stay['patientunitstayid']
    pid  = stay['uniquepid']
    hid  = stay['hospitalid']
    died = int(stay['hospitaldischargestatus'] == 'Expired')

    hosp_admit_day = stay['hospitaladmitoffset']  // MINS_PER_DAY  # ≤ 0
    icu_disch_day  = stay['unitdischargeoffset']  // MINS_PER_DAY  # ≥ 0
    hosp_disch_day = stay['hospitaldischargeoffset'] // MINS_PER_DAY

    for day in range(hosp_admit_day, hosp_disch_day + 1):
        is_icu = 1 if (0 <= day <= icu_disch_day) else 0
        rows.append({
            'patientunitstayid': uid,
            'uniquepid':         pid,
            'hospitalid':        hid,
            'day':               day,
            'is_icu':            is_icu,
            'died':              died,
            'last_day':          hosp_disch_day,
        })

daily = pd.DataFrame(rows)
print(f'Total patient-days:  {len(daily):,}')
print(f'ICU days:            {daily["is_icu"].sum():,}')
print(f'Non-ICU days:        {(daily["is_icu"] == 0).sum():,}')
print(f'Unique stays:        {daily["patientunitstayid"].nunique():,}')

---
## Section 3 — Vital Signs

We pull vitals from two places:

1. `vitalPeriodic` — high-frequency ICU monitoring (every few minutes), has MAP, SpO2, HR, RR, temperature
2. `nurseCharting` — covers the full hospital stay including before/after ICU, used for GCS and to fill in MAP/SpO2 gaps

For each patient-day we take the daily mean. When both sources have a value, we prefer `vitalPeriodic`.

In [ ]:
# ── vitalPeriodic: daily means ────────────────────────────────────────────────
vp = vitalPeriodic.copy()
vp['day'] = vp['observationoffset'] // MINS_PER_DAY

vp_daily = (
    vp.groupby(['patientunitstayid', 'day'])
    .agg(
        map_vp   =('systemicmean', 'mean'),
        spo2_vp  =('sao2',         'mean'),
        hr_vp    =('heartrate',    'mean'),
        rr_vp    =('respiration',  'mean'),
        temp_vp  =('temperature',  'mean'),
    )
    .reset_index()
)
print('vp_daily shape:', vp_daily.shape)
print('Non-null rates:')
print(vp_daily[['map_vp','spo2_vp','hr_vp','rr_vp','temp_vp']].notna().mean().round(3))

In [ ]:
# ── nurseCharting: MAP, SpO2, GCS ─────────────────────────────────────────────
nc = nurseCharting.copy()
nc['day'] = nc['nursingchartoffset'] // MINS_PER_DAY

# MAP from nurse charting
map_nc = nc[
    nc['nursingchartcelltypevallabel'].isin(['MAP (mmHg)', 'Arterial Line MAP (mmHg)'])
].copy()
map_nc['val'] = pd.to_numeric(map_nc['nursingchartvalue'], errors='coerce')
map_nc = map_nc[(map_nc['val'] > 0) & (map_nc['val'] < 200)]  # plausible range
map_nc_daily = (
    map_nc.groupby(['patientunitstayid', 'day'])['val']
    .mean().reset_index().rename(columns={'val': 'map_nc'})
)

# SpO2 from nurse charting
spo2_nc = nc[
    nc['nursingchartcelltypevallabel'].isin(['O2 Saturation', 'SpO2'])
].copy()
spo2_nc['val'] = pd.to_numeric(spo2_nc['nursingchartvalue'], errors='coerce')
spo2_nc = spo2_nc[(spo2_nc['val'] > 50) & (spo2_nc['val'] <= 100)]
spo2_nc_daily = (
    spo2_nc.groupby(['patientunitstayid', 'day'])['val']
    .mean().reset_index().rename(columns={'val': 'spo2_nc'})
)

# GCS total score (direct entry, not sub-components)
gcs_nc = nc[nc['nursingchartcelltypevallabel'] == 'Glasgow coma score'].copy()
gcs_nc['val'] = pd.to_numeric(gcs_nc['nursingchartvalue'], errors='coerce')
gcs_nc = gcs_nc[(gcs_nc['val'] >= 3) & (gcs_nc['val'] <= 15)]
gcs_daily = (
    gcs_nc.groupby(['patientunitstayid', 'day'])['val']
    .mean().reset_index().rename(columns={'val': 'gcs_total'})
)

print(f'map_nc_daily:  {len(map_nc_daily):,} rows')
print(f'spo2_nc_daily: {len(spo2_nc_daily):,} rows')
print(f'gcs_daily:     {len(gcs_daily):,} rows')

In [ ]:
# ── Merge vitals: prefer vitalPeriodic, fall back to nurseCharting ─────────────
vitals_daily = vp_daily.merge(map_nc_daily,  on=['patientunitstayid', 'day'], how='outer')
vitals_daily = vitals_daily.merge(spo2_nc_daily, on=['patientunitstayid', 'day'], how='outer')
vitals_daily = vitals_daily.merge(gcs_daily,     on=['patientunitstayid', 'day'], how='outer')

# Combine: prefer vitalPeriodic where available
vitals_daily['map']        = vitals_daily['map_vp'].combine_first(vitals_daily['map_nc'])
vitals_daily['spo2']       = vitals_daily['spo2_vp'].combine_first(vitals_daily['spo2_nc'])
vitals_daily['heart_rate'] = vitals_daily['hr_vp']
vitals_daily['resp_rate']  = vitals_daily['rr_vp']
vitals_daily['temperature']= vitals_daily['temp_vp']

vitals_daily = vitals_daily[[
    'patientunitstayid', 'day',
    'map', 'spo2', 'heart_rate', 'resp_rate', 'temperature', 'gcs_total'
]]

print('vitals_daily shape:', vitals_daily.shape)
print('Columns:', list(vitals_daily.columns))
vitals_daily.head(5)

---
## Section 4 — Lab Values

Lactate comes from the `lab` table where `labname == 'lactate'`. Same as MIMIC — we forward-fill within each stay since labs aren't drawn every day.

In [ ]:
# Lactate from lab table
lac = lab[lab['labname'] == 'lactate'].copy()
lac['day'] = lac['labresultoffset'] // MINS_PER_DAY
lac = lac[lac['labresult'].notna() & (lac['labresult'] > 0)]

labs_daily_raw = (
    lac.groupby(['patientunitstayid', 'day'])['labresult']
    .mean().reset_index().rename(columns={'labresult': 'lactate'})
)

print(f'Lactate rows: {len(lac):,}')
print(f'labs_daily_raw shape: {labs_daily_raw.shape}')
print(f'Unique stays with lactate: {labs_daily_raw["patientunitstayid"].nunique():,}')
labs_daily_raw.head(5)

In [ ]:
# Forward-fill lactate within each stay onto the full daily grid
patient_days = daily[['patientunitstayid', 'day']].drop_duplicates().copy()

labs_daily = patient_days.merge(labs_daily_raw, on=['patientunitstayid', 'day'], how='left')
labs_daily = labs_daily.sort_values(['patientunitstayid', 'day'])
labs_daily['lactate'] = labs_daily.groupby('patientunitstayid')['lactate'].ffill()

print('Missingness after forward-fill:')
print(f'  lactate: {labs_daily["lactate"].isna().mean():.1%}')

---
## Section 5 — Merging Everything Together

Join the daily backbone with vitals and labs. Then two-pass imputation:
1. Forward-fill within each stay
2. Median imputation for anything still missing

In [ ]:
# Start from the daily backbone
features = daily.copy()
features = features.merge(vitals_daily, on=['patientunitstayid', 'day'], how='left')
features = features.merge(labs_daily,   on=['patientunitstayid', 'day'], how='left')

feature_cols = ['map', 'spo2', 'gcs_total', 'lactate',
                'heart_rate', 'resp_rate', 'temperature']

print('Merged features shape:', features.shape)
print('\n── Missingness BEFORE imputation ──')
print(features[feature_cols].isna().mean().round(3))

In [ ]:
# Pass 1: forward-fill within each ICU stay
features = features.sort_values(['patientunitstayid', 'day'])
features[feature_cols] = (
    features.groupby('patientunitstayid')[feature_cols].ffill()
)

print('── Missingness AFTER forward-fill ──')
print(features[feature_cols].isna().mean().round(3))

In [ ]:
# Pass 2: median imputation for remaining NaNs
medians = features[feature_cols].median()
print('Medians used for imputation:')
print(medians.round(2))

features[feature_cols] = features[feature_cols].fillna(medians)

print('\n── Missingness AFTER median imputation ──')
print(features[feature_cols].isna().mean().round(3))
print('\nFeatures shape:', features.shape)

---
## Section 6 — State Discretization

Same clinical thresholds as before (SOFA-aligned):

| Variable | Bin 0 (worst) | Bin 1 | Bin 2 (normal) |
|---|---|---|---|
| MAP (mmHg) | < 65 | 65–100 | > 100 |
| SpO2 (%) | < 90 | 90–95 | ≥ 95 |
| GCS (3–15) | 3–8 | 9–12 | 13–15 |
| Lactate (mmol/L) | ≥ 4 | 2–4 | < 2 |

State index: `state = MAP_bin × 27 + SpO2_bin × 9 + GCS_bin × 3 + Lactate_bin`

82 states total: 81 live + 1 death (state 81).

In [ ]:
# ── Bin the four primary SOFA-aligned variables ───────────────────────────────
features['map_bin'] = pd.cut(
    features['map'], bins=[-np.inf, 65, 100, np.inf], labels=[0, 1, 2]
).astype(int)

features['spo2_bin'] = pd.cut(
    features['spo2'], bins=[-np.inf, 90, 95, np.inf], labels=[0, 1, 2]
).astype(int)

features['gcs_bin'] = pd.cut(
    features['gcs_total'], bins=[2, 8, 12, 15], labels=[0, 1, 2]
).astype(int)

# Lactate: higher = worse → invert labels
features['lactate_bin'] = pd.cut(
    features['lactate'], bins=[-np.inf, 2, 4, np.inf], labels=[2, 1, 0]
).astype(int)

# ── Secondary binned variables (for visualization; not used in state index) ───
def bin_hr(hr):
    if hr < 40 or hr > 130: return 0
    elif (40 <= hr < 60) or (100 < hr <= 130): return 1
    else: return 2

def bin_rr(rr):
    if rr < 10 or rr > 25: return 0
    elif (10 <= rr < 12) or (20 < rr <= 25): return 1
    else: return 2

def bin_temp(t):
    if t < 35 or t > 39.5: return 0
    elif (35 <= t < 36.5) or (38 < t <= 39.5): return 1
    else: return 2

features['hr_bin']   = features['heart_rate'].apply(bin_hr)
features['rr_bin']   = features['resp_rate'].apply(bin_rr)
features['temp_bin'] = features['temperature'].apply(bin_temp)

all_bins = {
    'map_bin':     ('MAP',         ['shock(<65)', 'normal(65-100)', 'hypertensive(>100)']),
    'spo2_bin':    ('SpO2',        ['critical(<90)', 'low(90-95)', 'normal(≥95)']),
    'gcs_bin':     ('GCS',         ['severe(3-8)', 'moderate(9-12)', 'normal(13-15)']),
    'lactate_bin': ('Lactate',     ['shock(≥4)', 'elevated(2-4)', 'normal(<2)']),
    'hr_bin':      ('Heart Rate',  ['dangerous', 'borderline', 'normal(60-100)']),
    'rr_bin':      ('Resp Rate',   ['dangerous', 'borderline', 'normal(12-20)']),
    'temp_bin':    ('Temperature', ['abnormal', 'borderline', 'normal(36.5-38°C)']),
}

print(f'{"Variable":<14}  Bin 0 (worst)  Bin 1 (abnormal)  Bin 2 (normal)')
print('-' * 65)
for col, (name, _) in all_bins.items():
    counts = features[col].value_counts().sort_index()
    c0, c1, c2 = counts.get(0, 0), counts.get(1, 0), counts.get(2, 0)
    print(f'{name:<14}  {c0:>6} days       {c1:>6} days       {c2:>6} days')

In [ ]:
# ── Bin distributions plot ─────────────────────────────────────────────────────
bin_colors = ['#d9534f', '#f0ad4e', '#5cb85c']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('eICU Demo — patient-day distribution across bins (0=worst, 2=normal)',
             fontsize=13, y=1.01)

for ax, (col, (name, _)) in zip(axes.flat, all_bins.items()):
    counts = features[col].value_counts().sort_index()
    vals   = [counts.get(i, 0) for i in range(3)]
    bars   = ax.bar(range(3), vals, color=bin_colors, edgecolor='white', linewidth=0.8)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xticks(range(3))
    ax.set_xticklabels(['0\n(worst)', '1\n(abnormal)', '2\n(normal)'], fontsize=8)
    ax.set_ylabel('Patient-days')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.2 + 10)

# hide the 8th subplot (we only have 7 variables now — no creatinine in eICU)
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eicu_bin_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eicu_bin_distributions.png')

In [ ]:
# ── Composite state index ─────────────────────────────────────────────────────
features['state'] = (
    features['map_bin']     * 27 +
    features['spo2_bin']    *  9 +
    features['gcs_bin']     *  3 +
    features['lactate_bin']
)

N_STATES   = 82
DEATH_STATE = 81

# ── Mark death state on last day of fatal hospital stays ──────────────────────
died_mask = (features['died'] == 1) & (features['day'] == features['last_day'])
features.loc[died_mask, 'state'] = DEATH_STATE

print(f'Patients who died in hospital:  {features[died_mask]["patientunitstayid"].nunique():,}')
print(f'Rows assigned death state:      {died_mask.sum():,}')
print(f'State range: {features["state"].min()} – {features["state"].max()}')
print(f'\nUnique states observed: {features["state"].nunique()} / {N_STATES}')
print('\nTop 15 most common states:')
print(features['state'].value_counts().head(15))

---
## Section 7 — Transition Matrix

Same logic as the MIMIC pipeline — extract consecutive-day pairs within each stay, split by ICU flag, count, normalize. Rows with no observations get a uniform distribution as fallback. Death is absorbing.

In [ ]:
# Sort by stay then day; shift within each stay to get next-day state
feat_sorted = features.sort_values(['patientunitstayid', 'day']).reset_index(drop=True)
feat_sorted['state_next'] = (
    feat_sorted.groupby('patientunitstayid')['state'].shift(-1)
)

transitions = feat_sorted.dropna(subset=['state_next']).copy()
transitions['state']      = transitions['state'].astype(int)
transitions['state_next'] = transitions['state_next'].astype(int)

print(f'Total transition pairs: {len(transitions):,}')
print(f'  ICU transitions:      {transitions["is_icu"].sum():,}')
print(f'  Non-ICU transitions:  {(1 - transitions["is_icu"]).sum():,}')

In [ ]:
# ── Build count matrices ───────────────────────────────────────────────────────
count_icu    = np.zeros((N_STATES, N_STATES), dtype=np.float64)
count_nonicu = np.zeros((N_STATES, N_STATES), dtype=np.float64)

for _, row in transitions.iterrows():
    s, sn, icu = int(row['state']), int(row['state_next']), int(row['is_icu'])
    if icu == 1:
        count_icu[s, sn]    += 1
    else:
        count_nonicu[s, sn] += 1

print('ICU    total counts:', count_icu.sum())
print('NonICU total counts:', count_nonicu.sum())

In [ ]:
# ── Row-normalise ──────────────────────────────────────────────────────────────
def normalise_rows(count_mat, n_states):
    prob = count_mat.copy()
    row_sums = prob.sum(axis=1, keepdims=True)
    nonzero  = (row_sums.squeeze() > 0)
    prob[nonzero]  = prob[nonzero] / row_sums[nonzero]
    prob[~nonzero] = 1.0 / n_states  # uniform fallback
    return prob, (~nonzero).sum()

P_icu,    icu_zeros    = normalise_rows(count_icu,    N_STATES)
P_nonicu, nonicu_zeros = normalise_rows(count_nonicu, N_STATES)

# Enforce absorbing death state
for P in (P_icu, P_nonicu):
    P[DEATH_STATE, :]             = 0.0
    P[DEATH_STATE, DEATH_STATE]   = 1.0

print(f'P_icu    — uniform rows (no data): {icu_zeros}')
print(f'P_nonicu — uniform rows (no data): {nonicu_zeros}')
print(f'P_icu    row-sum: min={P_icu.sum(1).min():.4f}, max={P_icu.sum(1).max():.4f}')
print(f'P_nonicu row-sum: min={P_nonicu.sum(1).min():.4f}, max={P_nonicu.sum(1).max():.4f}')

In [ ]:
# ── Save matrices ─────────────────────────────────────────────────────────────
np.save(os.path.join(OUT_DIR, 'P_icu_eicu.npy'),    P_icu)
np.save(os.path.join(OUT_DIR, 'P_nonicu_eicu.npy'), P_nonicu)
print('Saved P_icu_eicu.npy and P_nonicu_eicu.npy to', OUT_DIR)

---
## Section 8 — Plots

In [ ]:
# States with at least one observed transition
observed_states = np.where(
    (count_icu.sum(axis=1) + count_nonicu.sum(axis=1)) > 0
)[0]
print(f'States with ≥1 transition: {len(observed_states)} / {N_STATES}')

In [ ]:
# ── P_icu heatmap ─────────────────────────────────────────────────────────────
obs = observed_states
tick_step = max(1, len(obs) // 12)

for P, fname, title in [
    (P_icu,    'eicu_heatmap_P_icu.png',    'P_icu'),
    (P_nonicu, 'eicu_heatmap_P_nonicu.png', 'P_nonicu'),
]:
    P_obs  = P[np.ix_(obs, obs)]
    data_log = np.log10(np.where(P_obs > 0, P_obs, 1e-6))
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(data_log, aspect='auto', cmap='YlOrRd', vmin=-6, vmax=0)
    ax.set_title(f'{title} — eICU Demo (log₁₀ scale)\nShowing {len(obs)} observed states',
                 fontsize=13)
    ax.set_xlabel('Next-day state'); ax.set_ylabel('Current-day state')
    ax.set_xticks(range(0, len(obs), tick_step))
    ax.set_xticklabels(obs[::tick_step], rotation=45, fontsize=7)
    ax.set_yticks(range(0, len(obs), tick_step))
    ax.set_yticklabels(obs[::tick_step], fontsize=7)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('log₁₀(probability)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=120)
    plt.show()
    print('Saved:', fname)

In [ ]:
# ── State distribution bar chart ───────────────────────────────────────────────
state_dist = features['state'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(state_dist.index, state_dist.values,
       color=sns.color_palette('viridis', len(state_dist)), width=0.8)
ax.axvline(DEATH_STATE, color='red', linestyle='--', linewidth=1.5,
           label=f'Death state ({DEATH_STATE})')
ax.set_xlabel('State index')
ax.set_ylabel('Number of patient-days')
ax.set_title('eICU Demo — distribution of patient-days across health states')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eicu_state_distribution.png'), dpi=120)
plt.show()
print('Saved: eicu_state_distribution.png')

In [ ]:
# ── Self-loop comparison: ICU vs Non-ICU ──────────────────────────────────────
top10 = (
    features[features['state'] != DEATH_STATE]['state']
    .value_counts().head(10).index.tolist()
)

icu_self    = [P_icu[s, s]    for s in top10]
nonicu_self = [P_nonicu[s, s] for s in top10]

x, width = np.arange(len(top10)), 0.35
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width/2, icu_self,    width, label='ICU',     color='steelblue',  alpha=0.85)
b2 = ax.bar(x + width/2, nonicu_self, width, label='Non-ICU', color='darkorange', alpha=0.85)
ax.set_xlabel('State index (top 10 most visited)')
ax.set_ylabel('Self-loop probability P[s,s]')
ax.set_title('eICU Demo — Probability of remaining in same state: ICU vs Non-ICU')
ax.set_xticks(x)
ax.set_xticklabels([f'State {s}' for s in top10], rotation=30, ha='right')
ax.set_ylim(0, 1.1); ax.legend()
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'eicu_self_loop_comparison.png'), dpi=120)
plt.show()
print('Saved: eicu_self_loop_comparison.png')

---
## Section 9 — Summary

In [ ]:
n_stays      = features['patientunitstayid'].nunique()
n_patients   = features['uniquepid'].nunique()
n_total_trans= len(transitions)
n_icu_trans  = int(transitions['is_icu'].sum())
n_nonicu_trans= n_total_trans - n_icu_trans
n_states_obs = int((features['state'].value_counts() > 0).sum())
n_died       = int(features.loc[died_mask, 'patientunitstayid'].nunique())

summary = {
    'Dataset':                         'eICU-CRD Demo v2.0.1',
    'Unique patients (uniquepid)':      n_patients,
    'ICU unit stays':                   n_stays,
    'Total patient-days':              len(features),
    'Total transition pairs':          n_total_trans,
    '  ICU transitions':               n_icu_trans,
    '  Non-ICU transitions':           n_nonicu_trans,
    'Unique states observed':          n_states_obs,
    'State space size':                N_STATES,
    'ICU stays with death outcome':    n_died,
    'P_icu uniform rows (no data)':    int(icu_zeros),
    'P_nonicu uniform rows (no data)': int(nonicu_zeros),
}

print('=' * 55)
print('         eICU PIPELINE SUMMARY')
print('=' * 55)
for k, v in summary.items():
    print(f'  {k:<42} {str(v):>8}')
print('=' * 55)
print('\nOutput files:')
print(f'  P_icu_eicu.npy    : {P_icu.shape}  row-stochastic')
print(f'  P_nonicu_eicu.npy : {P_nonicu.shape}  row-stochastic')